# 편의점 신제품 데이터 정제 및 식품군 확정 EDA

이 노트북은 사용자의 검토 결과를 바탕으로 다음 단계를 수행합니다:
1. **[1단계] 명시적 제거:** 비식품/서비스로 확정된 단위 및 상품을 제거합니다.
2. **[2단계] 패턴 기반 필터링:** 제거된 상품에서 추출한 키워드로 '개, g, 입' 단위 내 비식품을 필터링합니다.
3. **[3단계] 규격화 대상 식별:** 남은 식품군 중 '캔' 등 재크롤링이 필요한 대상을 정리합니다.

In [2]:
def parse_formatted(s):
    try:
        import ast
        import re
        if not isinstance(s, str): return None
        s = s.strip()
        if s.startswith("{") and s.endswith("}"):
            s = s[1:-1].strip()
        
        idx = -1
        split_len = 0
        if ']\\\": [' in s: # JSON 이스케이프 대응
            idx = s.find(']\\\": [')
            split_len = 4
        elif ']\": [' in s:
            idx = s.find(']\": [')
            split_len = 4
        elif ']": [' in s:
            idx = s.find(']": [')
            split_len = 4
        elif "]: [" in s:
            idx = s.find("]: [")
            split_len = 2
        
        if idx == -1: return None
        
        k_str = s[:idx+1].strip()
        v_str = s[idx+split_len:].strip()
        
        if k_str.startswith('"'): k_str = k_str[1:]
        if k_str.endswith('"'): k_str = k_str[:-1]
        if v_str.startswith(':'): v_str = v_str[1:].strip()
        
        def py_fix(txt):
            txt = re.sub(r'\\bnull\\b', 'None', txt)
            return txt.replace('\\\\\"', '"')
        
        return list(ast.literal_eval(py_fix(k_str))), ast.literal_eval(py_fix(v_str))
    except:
        return None

[로드 완료] GS25_단일: 560행
[로드 완료] CU다중: 1901행
[로드 완료] 세븐다중: 1342행
[로드 완료] CU단일: 505행
[로드 완료] 세븐단일: 291행
[로드 완료] GS25다중: 1336행

총 6개 파일 로드 완료 (전체: 5935건)
모든 필수 파일(6개)이 정상적으로 로드되었습니다.


## [1단계] 명시적 제거 (Cleanup)
검토 결과에 따라 확정된 비식품 단위 및 상품을 제거합니다.

In [28]:
# 1. 제거 확정 단위
remove_units = ['정', 'p', '캡슐', '개월']

# 2. 제거 확정 특정 상품 키워드 (리포트 기반)
remove_items = [
    '택배', '트로피', '리프트', '렌탈', '이용권', '카밍패드', '엽서set', 
    '리무버오일패드', '비타민C', '이뮨샷', '파니니카드', '생리대', '천연펄프', 
    '데코', '컬렉션 카드', '스케이트 삭스', '마스크', '반바지', '바디피트'
]

def is_explicit_remove(row):
    if row['detected_unit'] in remove_units: return True
    name = str(row['p_name'])
    for item in remove_items:
        if item in name: return True
    return False

df_total['is_removed'] = df_total.apply(is_explicit_remove, axis=1)
df_cleaned = df_total[~df_total['is_removed']].copy()

print(f"명시적 제거 후 남은 데이터: {len(df_cleaned)}건 (제거됨: {df_total['is_removed'].sum()}건)")

명시적 제거 후 남은 데이터: 5859건 (제거됨: 76건)


## [2단계] 패턴 기반 필터링 (Criterion Filtering)
제거된 상품들의 특성을 활용해 '개, g, 입' 단위 내에 숨어있는 비식품을 추가로 걸러냅니다.

In [29]:
# '개', 'g', '입' 단위 상품군만 추출
target_units = ['개', 'g', '입', '봉', '구', '장']
df_targets = df_cleaned[df_cleaned['detected_unit'].isin(target_units)].copy()

# 필터링 기준: 비식품 의심 키워드 (Sheet2 및 검토 결과 기반 보강)
filter_keywords = [
    '카드', '굿즈', '에디션', '장난감', '피규어', '인형', '키링', '세트', '교통카드', '티켓', '복권',
    '다이어리', '플래너', '스티커', '포카', '앨범', '응원봉', '캘린더', '포스터', '마그넷',
    '비타민', '영양제', '오메가3', '유산균', '루테인', '콜라겐', '밀크씨슬', '멀티비타민',
    '리프트권', '이용권', '렌탈권', '패드', '마스크', '생리대', '팬티라이너', '반바지', '양말',
    '택배', '트로피', '메달', '코인', '금도금', '은트로피'
]

def check_suspicious(row):
    text = str(row['p_name']) + " " + " ".join([str(x) for x in row['p_attrs']] if row['p_attrs'] else [])
    for kw in filter_keywords:
        if kw in text: return True
    return False

df_targets['is_suspicious'] = df_targets.apply(check_suspicious, axis=1)

print(f"대상 단위({target_units}) 상품 중 의심 항목: {df_targets['is_suspicious'].sum()}건")
display(df_targets[df_targets['is_suspicious']][['p_name', 'p_cap', 'p_attrs', 'source_file']].head(10))


import os
ROOT = r"C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework"
out_path = os.path.join(ROOT, "eda", "filter_편의점_instagram.xlsx")
df_targets[df_targets['is_suspicious']][['p_name', 'p_cap', 'p_attrs', 'source_file']].to_excel(out_path, index=False)
print(f"결과가 엑셀로 저장되었습니다: {out_path}")

대상 단위(['개', 'g', '입', '봉', '구', '장']) 상품 중 의심 항목: 394건


,p_name,p_cap,p_attrs,source_file
19,이달의도시락1월(갓성비편),381g,"[불백, 도시락, CJ, 목우촌, 50% 할인, 카드 할인, 한 끼 완성, 가성비]",GS25_단일
21,몽모 키링세트 3종,3개,"[초콜릿, 달콤함, 스윗, 몽모, 한정수량, 사전예약]",GS25_단일
22,미니 눈오리 집게 키링 캔디,10G,"[캔디, 눈오리, 겨울]",GS25_단일
31,그랑크뤼 와인 9종,3개,"[와인, 삼성카드, 샤또와인, 팔머, 린치바쥐, 딸보, 샤스, 스플린, 까농, 로장...",GS25_단일
119,오쏘몰 이뮨,3개,"[비타민, 영양소, 액상, 정제, 건강기능식품, 오쏘몰, 유로모니터 인터내셔널, 론...",GS25_단일
129,하루엔진올인원,3개,"[비타민, 미네랄, 오메가3, 프로바이오틱스, 건강기능식품, 삼진제약, 동서바이오팜...",GS25_단일
181,운세부적키링팝핑캔디,3개,"[콜라 맛, 팝핑 캔디, 캔디류, 포스텔러 까리나, 히든이벤트, 운세, 행운]",GS25_단일
194,블랙춘 기획세트,3개,"[과자, 스낵, 상품권, 할인, 기획전, 한정수량, 명절, 선물]",GS25_단일
208,청명주 세트,3개,"[약주, 탁주, 전통주, 맑음, 깔끔함, 깊음, 한영석, 누룩명인, 사전예약, 한정수량]",GS25_단일
219,버터베어 굿즈 빼빼로 기획세트,1개,"[빼빼로, 버터베어, 시즌한정, 빼빼로데이]",GS25_단일


결과가 엑셀로 저장되었습니다: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\eda\filter_편의점_instagram.xlsx


## [3단계] 규격화 대상 식별 및 재크롤링 준비
필터링을 통과한 식품군 중 '캔' 등 용량 보정이 필요한 대상을 정리합니다.

In [30]:
# [추가] 수동 검수 결과(살려 리스트) 반영
filter_path = "../eda/filter_편의점_instagram.xlsx"
if os.path.exists(filter_path):
    df_filter = pd.read_excel(filter_path)
    keep_col = '살려' if '살려' in df_filter.columns else df_filter.columns[-1]
    keep_names = df_filter[df_filter[keep_col].notna()]['p_name'].tolist()
    print(f"수동 유지(살려) 대상: {len(keep_names)}건 로드 완료")
else:
    keep_names = []

# 최종 식품군: (비식품이 아니라고 판별됨) OR (수동 검수에서 살리기로 확정됨)
is_suspicious_idx = df_targets[df_targets['is_suspicious']].index
df_final_food = df_cleaned[
    (~df_cleaned.index.isin(is_suspicious_idx)) |
    (df_cleaned['p_name'].isin(keep_names))
]

# g/ml/kg/l 이외의 모든 단위 → 규격화 필요 대상
PURE_UNITS = {'g', 'ml', 'kg', 'l'}
recrawl_targets = df_final_food[
    df_final_food['detected_unit'].notna() &
    ~df_final_food['detected_unit'].isin(PURE_UNITS)
]

print(f"--- 재크롤링 및 용량 규격화 필요 대상: {len(recrawl_targets)}건 ---")
print("단위 분포:")
print(recrawl_targets['detected_unit'].value_counts().to_string())
display(recrawl_targets[['p_name', 'p_price', 'p_cap', 'source_file']].head(15))

# [업데이트] 원본 엑셀 파일에 'Y' 플래그 부여
for key, path in FILES.items():
    df_orig = pd.read_excel(path)
    target_names = recrawl_targets[recrawl_targets['source_file'] == key]['p_name'].tolist()

    updated_count = 0
    for idx, row in df_orig.iterrows():
        parsed = parse_formatted(row['formatted_output'])
        if parsed and parsed[0][0] in target_names:
            df_orig.at[idx, '재추출_가격용량'] = 'Y'
            updated_count += 1

    if updated_count > 0:
        df_orig.to_excel(path, index=False)
        print(f"[{key}] {updated_count}개 상품에 'Y' 플래그 부여 완료.")

수동 유지(살려) 대상: 394건 로드 완료
--- 재크롤링 및 용량 규격화 필요 대상: 3035건 ---
단위 분포:
detected_unit
개      2777
입        91
캔        53
조각       24
매        19
병        15
팩         7
t         7
알         6
봉         6
개입        6
구         4
인분        4
ml캔       4
일         3
봉입        2
g팩        2
ml병       2
입팩        1
종         1
장         1


,p_name,p_price,p_cap,source_file
1,데이지에일,12000.0,3캔,GS25_단일
2,데이지에일,12000.0,3캔,GS25_단일
3,데이지에일,12000.0,3캔,GS25_단일
6,말벡 와인,18900.0,3개,GS25_단일
7,로제불닭볶음면,1800.0,3개,GS25_단일
9,기네스 나이트로서지 디바이스,46000.0,1개,GS25_단일
13,깨찰빵(초코),2500.0,3개,GS25_단일
16,PLAVE 베이커리 5종,2500.0,3개,GS25_단일
18,에드워드리 K-체다치즈쫄떡볶이,3900.0,3개,GS25_단일
21,몽모 키링세트 3종,14500.0,3개,GS25_단일


[GS25_단일] 297개 상품에 'Y' 플래그 부여 완료.
[CU다중] 783개 상품에 'Y' 플래그 부여 완료.
[세븐다중] 985개 상품에 'Y' 플래그 부여 완료.
[CU단일] 342개 상품에 'Y' 플래그 부여 완료.
[세븐단일] 7개 상품에 'Y' 플래그 부여 완료.
[GS25다중] 670개 상품에 'Y' 플래그 부여 완료.


## [4단계] 데이터 품질 및 Pipeline 3 결과 정밀 분석

이 단계에서는 Pipeline 3 실행 후의 전체적인 데이터 상태를 점검합니다. 특히 **상품명이 NaN으로 표시되는 데이터**가 파이프라인 중 어디에서 소실되었는지(파싱 버그 vs 추출 실패)를 중점적으로 확인합니다.

In [ ]:
# 1. 데이터 로드 및 기초 품질 점검
all_rows = []
for key, path in FILES.items():
    if os.path.exists(path):
        df_tmp = pd.read_excel(path)
        df_tmp['source_file'] = key
        all_rows.append(df_tmp)

df_all = pd.concat(all_rows, ignore_index=True)

def check_raw_quality(row):
    raw = row.get('formatted_output')
    if pd.isna(raw) or str(raw).strip() == "":
        return '1. 원본 데이터 부재(LLM 추출 실패)'
    
    parsed = parse_formatted(raw)
    if parsed is None:
        return '2. 파싱 실패(Parser 버그 의심)'
    
    k, v = parsed
    name = k[0] if len(k) > 0 else None
    if name is None or str(name).lower() == 'nan' or str(name).strip() == "":
        return '3. 상품명 누락(LLM이 이름을 못 뽑음)'
    
    return '0. 정상'

df_all['quality_check'] = df_all.apply(check_raw_quality, axis=1)

print('=== [품질 점검] 전체 데이터 상태 ===')
qc_counts = df_all['quality_check'].value_counts().sort_index()
print(qc_counts)
print('=' * 40)

# 파싱 성공한 데이터에 대해서만 정보 추출
def extract_info_extended(row):
    parsed = parse_formatted(row['formatted_output'])
    if parsed:
        k, v = parsed
        return pd.Series([k[0] if len(k) > 0 else None,
                          k[1] if len(k) > 1 else None,
                          k[2] if len(k) > 2 else None])
    return pd.Series([None, None, None])

df_all[['p_name', 'p_price', 'p_cap']] = df_all.apply(extract_info_extended, axis=1)
df_all['detected_unit'] = df_all['p_cap'].apply(extract_unit)
df_all['is_unresolved'] = df_all.get('재추출_가격용량', '') == 'Y'

unresolved_normal = df_all[df_all['is_unresolved'] & (df_all['quality_check'] == '0. 정상')].shape[0]
print(f'\n정상 데이터 중 \'재추출_가격용량\' 잔존(미해결) 건수: {unresolved_normal}건')

In [ ]:
# 2. 파서를 통과하지 못한 데이터 상세 분석 (소실 데이터 검토)
problematic = df_all[df_all['quality_check'] != '0. 정상'].copy()

if len(problematic) > 0:
    print(f'=== 소실/오류 데이터 분석 (총 {len(problematic)}건) ===')
    # 파일별/오류유형별 분포
    summary = problematic.groupby(['source_file', 'quality_check']).size().unstack(fill_value=0)
    display(summary)
    
    print('\n▶ [파싱 실패/상품명 누락] 원본 샘플 (상위 10개)')
    # 실제 formatted_output이 어떻게 생겼는지 확인하여 버그 수정 힌트 확보
    display(problematic[['source_file', 'quality_check', 'formatted_output']].head(10))
else:
    print('모든 데이터가 정상적으로 파싱되었습니다.')

In [ ]:
# 3. 미해결 원인 분류 (정상 데이터 중 미해결 건만 대상)
df_un = df_all[(df_all['is_unresolved']) & (df_all['quality_check'] == '0. 정상')].copy()

if len(df_un) > 0:
    NON_FOOD_KW   = ['카드', '굿즈', '기프트', '트로피', '택배', '리프트', '프린팅', '이용권',
                      '렌탈', '티켓', '스티커', '앨범', '포카', '응원봉', '다이어리']
    MULTI_PACK_KW = ['세트', '기획', '묶음', '패키지']
    VAGUE_KW      = ['급식대가', '이벤트', '할인', '혜택', '추첨', '증정']

    def classify_reason(name):
        name_s = str(name)
        if any(kw in name_s for kw in NON_FOOD_KW): return '비식품(굿즈/서비스)'
        if any(kw in name_s for kw in MULTI_PACK_KW): return '묶음상품(단위불명)'
        if any(kw in name_s for kw in VAGUE_KW): return '모호한 마케팅 문구'
        return '크롤링 미매칭(재시도 가능)'

    df_un['reason'] = df_un['p_name'].apply(classify_reason)

    print(f'=== 미해결 원인 분석 (총 {len(df_un)}건) ===')
    print(df_un['reason'].value_counts())

    for reason in df_un['reason'].unique():
        subset = df_un[df_un['reason'] == reason][['p_name', 'p_cap', 'source_file']]
        print(f'\n[{reason}] {len(subset)}건 — 샘플 10개')
        display(subset.head(10).reset_index(drop=True))
else:
    print('미해결된 정상 데이터가 없습니다.')